In [ ]:
using Bessels, Plots, LinearAlgebra

const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
φ_b = 0.204

E_J = 37.12*2pi/2

#Environment
κ_b=5/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.1
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 4*g2

g = E_J*sin(ε_p)*φ_a^2*φ_b
ε_d_ratio = 2
params = (ε_p,E_J,ε_d_ratio,κ_b);

M = 100

A_0 = sqrt(ε_d / g)
B_0 = sqrt(2) * ε_d / κ_b

In [ ]:
ϕ_a = [A_0, 0]
ϕ_b = [0, -B_0]

# initial string in phase-space
Δs = 1 / (M-1)
s=0:Δs:1

A_init = [A_0*(1 - s_i) for s_i in s]
P_A_init = [A_0*s_i for s_i in s]
P_B_init = [2 * A_init[i]/A_0 * (A_init[i]/A_0-1)*B_0 for (i, s_i) in enumerate(s)]
B_init = [-P_B_init[i] for (i, s_i) in enumerate(s)]

In [ ]:
function H(A, B, P_A, P_B, params)
    (ε_p,E_J,ε_d_ratio,κ_b) = params
    g = E_J*sin(ε_p)*φ_a^2*φ_b
    ε_d = ε_d_ratio*g
    return -sqrt(2)*g*A*P_A*B +
           (g*(A^2 + P_A^2)/sqrt(2) - ε_d/sqrt(2) - κ_b*B/2)*P_B -
           κ_b*P_B^2/2
end
function JgradH(x, params)
    A, P_A, B, P_B = x
    (ε_p,E_J,ε_d_ratio,κ_b) = params
    g = E_J*sin(ε_p)*φ_a^2*φ_b
    ε_d = ε_d_ratio*g
    dA   = -sqrt(2)*g*(P_A*B-A*P_B)
    dB   = -(κ_b/2)*P_B-sqrt(2)*g*A*P_A
    dP_A = -sqrt(2)*g*A*B + sqrt(2)*g*P_A*P_B
    dP_B = g*(A^2 + P_A^2)/sqrt(2) - ε_d/sqrt(2) - κ_b*B/2 - κ_b*P_B
    return dP_A, dP_B, -dA, -dB
end

In [ ]:
function fd_jacobian(F, z; h=1e-8)
    n = length(z)
    J = zeros(n, n)
    F0 = F(z)
    for j in 1:n
        zp = copy(z)
        dz = h * max(1.0, abs(z[j]))
        zp[j] += dz
        J[:, j] = (F(zp) - F0) / dz
    end
    return J[1:end-1, 1:end-1]
end

In [ ]:
function F(z)
    X = reshape(z[1:4*(M-2)], (4, M-2))
    α = z[4*(M-2)+1:end]

    function get_Xi(i)
        if i == 1
            return x0
        elseif i == M
            return xM
        else
            return X[:, i-1]
        end
    end

    F_test = zeros(4*(M-2) + (M-1))

    idx = 1

    # flow constraints on segments i = 1, ..., M-2
    for i in 1:M-2
        xi   = get_Xi(i)
        xip1 = get_Xi(i+1)

        v = JgradH(xi, params)

        F_test[idx:idx+3] .= xip1 .- xi .- α[i] .* v
        idx += 4
    end

    # energy constraints on points i = 2, ..., M
    for i in 2:M
        xi = get_Xi(i)
        A, P_A, B, P_B = xi
        F_test[idx] = H(A, B, P_A, P_B, params)
        idx += 1
    end

    return F_test
end

In [ ]:
x0 = [A_init[1], P_A_init[1], B_init[1], P_B_init[1]]
xM = [A_init[M], P_A_init[M], B_init[M], P_B_init[M]]
x_guess = zeros(4*(M-2))
for i in 0:M-3
    x_guess[4*i+1] = A_init[i+2]
    x_guess[4*i+2] = P_A_init[i+2]
    x_guess[4*i+3] = B_init[i+2]
    x_guess[4*i+4] = P_B_init[i+2]
end
α0 = 0.1 * ones(M-1)
z0 = vcat(x_guess, α0)
F_test = F(z0)

In [ ]:
J = fd_jacobian(F, z0; h=1e-4)

In [ ]:
J[:, end] == zeros(490)

In [ ]:
function newton_solver(F, z0; tol=1e-10, maxiter=30, verbose=true)
    z = copy(z0)

    for k in 1:maxiter
        R = F(z)[1:end-1]
        nr = norm(R)

        if verbose
            println("iter = $k   ‖R‖ = $nr")
        end

        if nr < tol
            return z, true
        end

        J = fd_jacobian(F, z)
        δz = -J \ R
        
        λ = 1.0
        display("OK")
        ztrial = z .+ λ .* δz
        
        Rtrial = F(ztrial)
        
        while norm(Rtrial) > (1 - 1e-4 * λ) * nr && λ > 1e-6
            λ *= 0.5
            ztrial = z .+ λ .* δz
            Rtrial = F(ztrial)
        end

        z .= ztrial
    end

    return z, false
end

In [ ]:
newton_solver(F, z0)